# Introdução à Análise Fatorial Exploratória (AFE)

Diferente do PCA (que apenas comprime dados matematicamente), a **Análise Fatorial** é um modelo causal. Ela parte da teoria de que existe uma "Variável Latente" (que não podemos observar diretamente) que está *causando* o comportamento das variáveis que observamos (neste caso, as notas).

### Objetivo
Identificar construtos latentes no desempenho acadêmico. Temos notas de 4 disciplinas. Será que conseguimos extrair 2 fatores latentes (ex: "Perfil Exatas" e "Perfil Humanas")?

### O que será feito neste Step-by-Step:
1. **Preparação**: Importação do `factor_analyzer`.
2. **Dataset**: Dados simulados de notas acadêmicas.
3. **Testes de Adequação (KMO e Bartlett)**: A matemática permite fazer FA nestes dados?
4. **Extração de Fatores**: Quantos fatores latentes existem?
5. **Rotação (Varimax)**: O segredo para facilitar a interpretação dos fatores.
6. **Comunalidades**: O quanto cada variável é explicada pelos fatores descobertos.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from factor_analyzer import FactorAnalyzer
from factor_analyzer.factor_analyzer import calculate_bartlett_sphericity, calculate_kmo

import warnings
warnings.filterwarnings("ignore")

### Passo 2: O Dataset de Notas

Alunos com boas notas em `Finanças` tendem a ir bem em `Atuária`. Alunos que vão bem em `Marketing` tendem a ir bem em `Liderança`.

In [2]:
dados_notas = {
    'financas': [8.0, 7.5, 4.0, 9.0, 5.0, 8.5, 4.5, 9.5],
    'atuaria': [8.5, 7.0, 3.5, 9.5, 4.0, 9.0, 3.0, 8.5],
    'marketing': [5.0, 6.0, 9.0, 4.5, 8.5, 5.5, 9.5, 6.0],
    'lideranca': [5.5, 5.0, 8.5, 5.0, 9.0, 4.5, 9.0, 5.5]
}

df_notas = pd.DataFrame(dados_notas)
display(df_notas.head())

,financas,atuaria,marketing,lideranca
0,8.0,8.5,5.0,5.5
1,7.5,7.0,6.0,5.0
2,4.0,3.5,9.0,8.5
3,9.0,9.5,4.5,5.0
4,5.0,4.0,8.5,9.0


### Passo 3: Testes de Adequação da Amostra

Antes de rodar a Análise Fatorial, precisamos provar para a estatística que nossas variáveis possuem correlações suficientes entre si. Se não houver correlação, não há fator latente a ser descoberto.

* **Teste de Bartlett**: Testa se a matriz de correlação é igual a uma matriz identidade (variáveis totalmente independentes). Queremos um p-valor < 0.05.
* **Teste KMO (Kaiser-Meyer-Olkin)**: Mede a proporção da variância que pode ser variância comum. Valores > 0.6 são aceitáveis.

In [ ]:
# Teste de Bartlett
chi_square_value, p_value = calculate_bartlett_sphericity(df_notas)
print(f"Teste de Esfericidade de Bartlett (p-value): {p_value:.5f}")

# Teste KMO
kmo_all, kmo_model = calculate_kmo(df_notas)
print(f"KMO Geral da Amostra: {kmo_model:.3f}")

### Passo 4: Extração e Autovalores (Critério de Kaiser)

Quantos fatores devemos extrair? A regra clássica (Critério de Kaiser) diz que devemos reter fatores que possuem um **Autovalor (Eigenvalue) > 1**. Um fator com autovalor > 1 explica mais do que uma única variável isolada.

In [ ]:
# Inicializando o analisador apenas para pegar os autovalores
fa_inicial = FactorAnalyzer(n_factors=df_notas.shape[1], rotation=None)
fa_inicial.fit(df_notas)

ev, v = fa_inicial.get_eigenvalues()

plt.figure(figsize=(6,4))
plt.plot(range(1, df_notas.shape[1]+1), ev, marker='o', color='purple')
plt.axhline(y=1, color='r', linestyle='--', label='Critério de Kaiser (y=1)')
plt.title('Scree Plot - Autovalores')
plt.xlabel('Fatores')
plt.ylabel('Autovalor')
plt.legend()
plt.grid(True)
plt.show()

# Baseado no gráfico, extrairemos 2 fatores.

### Passo 5: Rotação Ortogonal (Varimax) e Cargas Fatoriais

A extração crua dos fatores muitas vezes gera "cargas" matemáticas difíceis de interpretar (uma mesma variável aparece associada aos dois fatores).
Aplicamos a **Rotação Varimax**. Geometricamente, ela gira os eixos do gráfico de fatores para maximizar a carga de uma variável em um fator e minimizá-la no outro, clareando a interpretação.

In [ ]:
# Ajustando o modelo final com 2 fatores e rotação Varimax
fa_final = FactorAnalyzer(n_factors=2, rotation='varimax')
fa_final.fit(df_notas)

# Obtendo as Cargas Fatoriais
cargas = pd.DataFrame(fa_final.loadings_, index=df_notas.columns, columns=['Fator 1', 'Fator 2'])

plt.figure(figsize=(8, 4))
sns.heatmap(cargas, annot=True, cmap='viridis', center=0)
plt.title('Cargas Fatoriais Pós-Rotação Varimax')
plt.show()

# Interpretação clara:
# O Fator 1 carrega pesadamente em Finanças e Atuária -> Podemos chamar de "Perfil Exatas"
# O Fator 2 carrega pesadamente em Marketing e Liderança -> Podemos chamar de "Perfil Humanas/Gestão"

### Passo 6: Comunalidades

A Comunalidade indica a proporção da variância de cada variável original que é explicada pelos Fatores Latentes que acabamos de extrair. Valores altos indicam que o modelo capturou bem o comportamento daquela variável.

In [ ]:
comunalidades = pd.DataFrame(fa_final.get_communalities(), index=df_notas.columns, columns=['Comunalidade'])
display(comunalidades)

# Variáveis com comunalidade perto de 1 estão perfeitamente representadas pelos 2 fatores extraídos.